# Cassava leaf disease classifier (Google Colab)

This notebook trains a **separate experimental cassava classifier** from the official TensorFlow Datasets `cassava` dataset. It does not use or overwrite the seven-crop model.

## Before running
1. Upload this notebook to Google Colab and open it.
2. In Colab, choose **Runtime → Change runtime type → T4 GPU** (or another available GPU). CPU also works, but will be slower.
3. Run the cells from top to bottom. Approve Google Drive access when prompted.
4. The first dataset and pretrained-weight downloads require internet access. The dataset is downloaded by TFDS inside Colab; nothing needs to be uploaded from your Windows computer.

By default, the training subset is **10 randomly selected images per class**, matching the current small-sample experiment request. The official validation and test splits are kept separate and are not used to fit the model. For a stronger later experiment, change `TRAIN_IMAGES_PER_CLASS` to `None` to use the complete official training split.

The five label names and split counts are read from TFDS metadata at runtime rather than assumed. Results from a small 10-per-class run are exploratory and are not evidence of reliable field diagnosis. Do not use predictions alone to decide treatment.

In [ ]:
%pip -q install "tensorflow-datasets>=4.9,<5" "scikit-learn>=1.4,<2"

In [ ]:
import csv
import json
import platform
import random
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
from google.colab import drive
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

SEED = 42
IMAGE_SIZE = (160, 160)
BATCH_SIZE = 32
EPOCHS = 10
TRAIN_IMAGES_PER_CLASS = 10  # Set to None to use the complete official training split.
DATA_DIR = "/content/tensorflow_datasets"
OUTPUT_DIR = Path("/content/drive/MyDrive/cassava_leaf_model")

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

drive.mount("/content/drive")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Python:", platform.python_version())
print("TensorFlow:", tf.__version__)
print("TensorFlow Datasets:", tfds.__version__)
print("Devices:", tf.config.list_physical_devices())
print("Artifacts will be saved to:", OUTPUT_DIR)

## Download and inspect the cassava dataset

TFDS prepares the published train, validation, and test splits in Colab storage. The first run downloads the dataset and may take several minutes. The metadata and observed split sizes are printed below.

In [ ]:
builder = tfds.builder("cassava", data_dir=DATA_DIR)
builder.download_and_prepare()
info = builder.info

if "label" not in info.features or not info.features["label"].names:
    raise ValueError("TFDS did not provide a named cassava label feature.")
CLASS_NAMES = list(info.features["label"].names)
SPLIT_SIZES = {name: split.num_examples for name, split in info.splits.items()}
if set(SPLIT_SIZES) != {"train", "validation", "test"}:
    raise ValueError(f"Unexpected TFDS splits: {SPLIT_SIZES}")
if len(CLASS_NAMES) != 5:
    raise ValueError(f"Expected five cassava classes, found {CLASS_NAMES}")

print("Dataset:", info.name, info.version)
print("Citation:", info.citation)
print("Class index mapping:")
for index, name in enumerate(CLASS_NAMES):
    print(f"  {index}: {name}")
print("Published split sizes:", SPLIT_SIZES)
print("Total images:", sum(SPLIT_SIZES.values()))

## Create the training input

When the 10-image setting is active, this cell deterministically samples 10 images per discovered class from the **training split only**. Validation and test examples are never added to training. Set the configuration to `None` to use the entire official training split.

In [ ]:
raw_train = builder.as_dataset(split="train", shuffle_files=False)

if TRAIN_IMAGES_PER_CLASS is None:
    train_data = raw_train.map(
        lambda item: (tf.image.resize(item["image"], IMAGE_SIZE), tf.cast(item["label"], tf.int32)),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    train_data = train_data.shuffle(SPLIT_SIZES["train"], seed=SEED, reshuffle_each_iteration=True)
    train_data = train_data.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    training_counts = None
else:
    rng = random.Random(SEED)
    selected_images = [[] for _ in CLASS_NAMES]
    seen = [0 for _ in CLASS_NAMES]
    shuffled_train = raw_train.shuffle(
        min(SPLIT_SIZES["train"], 1024), seed=SEED, reshuffle_each_iteration=False
    )
    for item in tfds.as_numpy(shuffled_train):
        label = int(item["label"])
        if len(selected_images[label]) < TRAIN_IMAGES_PER_CLASS:
            resized = tf.image.resize(item["image"], IMAGE_SIZE).numpy().astype(np.float32)
            selected_images[label].append(resized)
        seen[label] += 1
        if all(len(images) == TRAIN_IMAGES_PER_CLASS for images in selected_images):
            break

    missing = [CLASS_NAMES[i] for i, images in enumerate(selected_images) if len(images) < TRAIN_IMAGES_PER_CLASS]
    if missing:
        raise ValueError(f"Not enough training images to sample {TRAIN_IMAGES_PER_CLASS} per class: {missing}")
    train_images = np.stack([image for class_images in selected_images for image in class_images])
    train_labels = np.concatenate([
        np.full(len(class_images), class_index, dtype=np.int32)
        for class_index, class_images in enumerate(selected_images)
    ])
    train_data = tf.data.Dataset.from_tensor_slices((train_images, train_labels))
    train_data = train_data.shuffle(len(train_labels), seed=SEED, reshuffle_each_iteration=True)
    train_data = train_data.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    training_counts = {CLASS_NAMES[i]: len(images) for i, images in enumerate(selected_images)}

def make_eval_dataset(split_name):
    dataset = builder.as_dataset(split=split_name, shuffle_files=False)
    dataset = dataset.map(
        lambda item: (tf.image.resize(item["image"], IMAGE_SIZE), tf.cast(item["label"], tf.int32)),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

validation_data = make_eval_dataset("validation")
test_data = make_eval_dataset("test")
print("Training examples:", TRAIN_IMAGES_PER_CLASS * len(CLASS_NAMES) if TRAIN_IMAGES_PER_CLASS is not None else SPLIT_SIZES["train"])
print("Training examples per class:", training_counts if training_counts is not None else "full published training split")
print("Validation examples:", SPLIT_SIZES["validation"])
print("Held-out test examples:", SPLIT_SIZES["test"])

## Train MobileNetV2

The ImageNet backbone is frozen for this first experiment. Keras downloads the pretrained weights on the first run. Augmentation is inside the model and runs during training only. The best checkpoint is selected using validation loss; the test split remains untouched until final evaluation.

In [ ]:
augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal", seed=SEED),
        tf.keras.layers.RandomRotation(0.08, seed=SEED),
        tf.keras.layers.RandomZoom(0.1, seed=SEED),
    ],
    name="training_augmentation",
)
backbone = tf.keras.applications.MobileNetV2(
    input_shape=(*IMAGE_SIZE, 3), include_top=False, weights="imagenet"
)
backbone.trainable = False

inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3), name="image")
x = augmentation(inputs)
x = tf.keras.layers.Rescaling(1.0 / 127.5, offset=-1, name="mobilenet_scaling")(x)
x = backbone(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2, seed=SEED)(x)
outputs = tf.keras.layers.Dense(len(CLASS_NAMES), activation="softmax", name="predictions")(x)
model = tf.keras.Model(inputs, outputs, name="cassava_mobilenetv2")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

LOCAL_CHECKPOINT = Path("/content/best_cassava_model.keras")
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        LOCAL_CHECKPOINT, monitor="val_loss", save_best_only=True, verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True, verbose=1
    ),
]
started_at = datetime.now(timezone.utc)
start_time = time.perf_counter()
history = model.fit(
    train_data, validation_data=validation_data, epochs=EPOCHS,
    callbacks=callbacks, verbose=2,
)
training_seconds = time.perf_counter() - start_time
model = tf.keras.models.load_model(LOCAL_CHECKPOINT)
print(f"Training duration: {training_seconds / 60:.1f} minutes")

## Final held-out test evaluation

Run this cell once after training. It records accuracy, macro/weighted F1, per-class metrics, confusion matrix, and inference time.

In [ ]:
test_labels = np.concatenate([labels.numpy() for _, labels in test_data], axis=0)
inference_start = time.perf_counter()
test_scores = model.predict(test_data, verbose=0)
inference_seconds = time.perf_counter() - inference_start
test_predictions = np.argmax(test_scores, axis=1)
label_indices = list(range(len(CLASS_NAMES)))
class_report = classification_report(
    test_labels, test_predictions, labels=label_indices,
    target_names=CLASS_NAMES, output_dict=True, zero_division=0,
)
matrix = confusion_matrix(test_labels, test_predictions, labels=label_indices)
metrics = {
    "evaluated_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_name": info.name,
    "dataset_version": str(info.version),
    "class_names_in_index_order": CLASS_NAMES,
    "split_sizes": SPLIT_SIZES,
    "training_images_per_class": TRAIN_IMAGES_PER_CLASS,
    "training_examples": int(TRAIN_IMAGES_PER_CLASS * len(CLASS_NAMES)) if TRAIN_IMAGES_PER_CLASS is not None else SPLIT_SIZES["train"],
    "test_image_count": int(len(test_labels)),
    "accuracy": float(accuracy_score(test_labels, test_predictions)),
    "macro_f1": float(class_report["macro avg"]["f1-score"]),
    "weighted_f1": float(class_report["weighted avg"]["f1-score"]),
    "per_class": {name: class_report[name] for name in CLASS_NAMES},
    "confusion_matrix": matrix.tolist(),
    "confusion_matrix_axis_order": CLASS_NAMES,
    "inference_total_seconds": inference_seconds,
    "inference_milliseconds_per_image": inference_seconds * 1000 / len(test_labels),
    "training_duration_seconds": training_seconds,
    "confidence_note": "Softmax scores are not necessarily calibrated probabilities of correctness.",
    "research_limitations": [
        "This is an experimental cassava classifier, not a field-validated diagnosis system.",
        "When limited to 10 training images per class, performance may be substantially worse than training on the full dataset.",
        "The dataset labels and split metrics do not establish generalization to all farms, devices, or locations.",
    ],
}

print(f"Held-out test images: {len(test_labels)}")
print(f"Accuracy: {metrics['accuracy']:.4f}")
print(f"Macro F1: {metrics['macro_f1']:.4f}")
print(f"Weighted F1: {metrics['weighted_f1']:.4f}")
print(f"Inference: {metrics['inference_milliseconds_per_image']:.2f} ms/image")
print("Per-class precision / recall / F1:")
for name in CLASS_NAMES:
    entry = metrics["per_class"][name]
    print(f"  {name}: {entry['precision']:.3f} / {entry['recall']:.3f} / {entry['f1-score']:.3f} (n={int(entry['support'])})")

## Save deliverables to Google Drive

The model and companion metadata are saved together under `MyDrive/cassava_leaf_model/`. Download that directory or copy it to the application server before integration. Keep the model and `class_labels.json` paired: output index order must match the label mapping.

In [ ]:
class_mapping = {
    "task": "cassava_leaf_disease_classification",
    "dataset": info.name,
    "dataset_version": str(info.version),
    "class_count": len(CLASS_NAMES),
    "class_names": CLASS_NAMES,
    "class_to_index": {name: index for index, name in enumerate(CLASS_NAMES)},
    "index_to_class": {str(index): name for index, name in enumerate(CLASS_NAMES)},
}
model_path = OUTPUT_DIR / "cassava_model.keras"
labels_path = OUTPUT_DIR / "class_labels.json"
metrics_path = OUTPUT_DIR / "test_metrics.json"
history_path = OUTPUT_DIR / "training_history.json"
matrix_path = OUTPUT_DIR / "confusion_matrix.csv"

model.save(model_path)
labels_path.write_text(json.dumps(class_mapping, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
metrics_path.write_text(json.dumps(metrics, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
history_report = {
    "started_at_utc": started_at.isoformat(),
    "training_duration_seconds": training_seconds,
    "tensorflow_version": tf.__version__,
    "tensorflow_datasets_version": tfds.__version__,
    "architecture": "MobileNetV2 ImageNet pretrained frozen backbone",
    "image_size": list(IMAGE_SIZE),
    "batch_size": BATCH_SIZE,
    "maximum_epochs": EPOCHS,
    "epochs_completed": len(history.history.get("loss", [])),
    "random_seed": SEED,
    "training_images_per_class": TRAIN_IMAGES_PER_CLASS,
    "split_sizes": SPLIT_SIZES,
    "history": {key: [float(value) for value in values] for key, values in history.history.items()},
    "test_set_used_during_training": False,
}
history_path.write_text(json.dumps(history_report, indent=2) + "\n", encoding="utf-8")
with matrix_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["true_label", *CLASS_NAMES])
    for name, row in zip(CLASS_NAMES, matrix.tolist()):
        writer.writerow([name, *row])

# Reload once to check the saved .keras file before using it in an application.
reloaded_model = tf.keras.models.load_model(model_path)
assert reloaded_model.output_shape[-1] == len(CLASS_NAMES)
print("Saved and reloaded model:", model_path)
print("Class mapping:", labels_path)
print("Test metrics:", metrics_path)
print("Training history:", history_path)
print("Confusion matrix:", matrix_path)
print("Model file size (MiB):", round(model_path.stat().st_size / (1024 * 1024), 2))

## Predict one uploaded image (optional)

This cell opens a Colab upload dialog. The displayed score is a softmax output, not necessarily a calibrated probability. The model can only choose one of its cassava training labels and cannot reliably reject unknown images.

In [ ]:
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise ValueError("No image was uploaded.")
image_name = next(iter(uploaded))
image_bytes = tf.io.read_file(image_name)
image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
image = tf.image.resize(image, IMAGE_SIZE)
scores = reloaded_model.predict(tf.expand_dims(image, axis=0), verbose=0)[0]
predicted_index = int(np.argmax(scores))
print("Predicted cassava class:", CLASS_NAMES[predicted_index])
print("Model score (not calibrated probability):", float(scores[predicted_index]))